# Supervisor Experiment 2 -- Google Colab Runner

Thin Colab runner around the finalized, audited Exp2 script -- **no scientific
logic lives in this notebook**. Every cell below either verifies the
environment, wires up persistent storage/paths, or invokes the script
unchanged.

**Scientific source of truth (unchanged by this notebook):**
`experiments_prepared/supervisor_exp2_cifar100_5x20_rank32_vs_rankext160.py`

- CIFAR-100, 5x20 protocol, seed 42, 9 epochs
- SimpleAvg: rank 32, alpha 64, scaling 2.0
- RankExt: cumulative ranks `[32,64,96,128,160]`, scaling 2.0
- 8 canonical methods (simple_avg x4, rank_extension x4) -- no R8 code
- KD (T=2, weight=1), FactorOrth (lambda=50), calibration
  (`confidence_weighted_regime_grouped`), best-epoch selection, classifier
  handling, SimpleAvg dense-delta merge, and all metrics: byte-identical to
  the already-audited Experiment 2 (see
  `thesis_agent/reports/supervisor_exp2_preparation_audit.md`).

This notebook adds only two kinds of infrastructure, both already present in
the shared script itself (so cluster jobs are unaffected unless they opt in
via the same environment variables this notebook sets):

1. **Persistent Google Drive paths** via `EXP2_RESULTS_ROOT` (output
   directory) instead of the ephemeral, `/content`-relative default.
2. **Method-level resume** via `EXP2_RUN_TAG` (a stable run identifier reused
   across Colab reconnects) plus the script's own `method_results/*.json`
   atomic-write/skip-if-completed mechanism -- so a Colab disconnect does not
   force an already-completed method to be retrained.

See `thesis_agent/reports/supervisor_exp2_colab_sync_audit.md` for the full
cluster-vs-Colab equivalence audit.

**Run the cells in order, top to bottom.** The final "Run exact Experiment 2"
cell is the only one that starts real training -- everything before it is
setup/verification.

## 1. GPU / runtime verification

Fails clearly and stops here if no CUDA GPU is attached (Runtime -> Change runtime type -> Hardware accelerator -> GPU).

In [ ]:
import sys
import subprocess

print("Python version:", sys.version)

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available (torch.cuda.is_available()):", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not available. In Colab: Runtime -> Change runtime type -> "
        "Hardware accelerator -> GPU (or a paid tier's A100/V100/L4), then "
        "Runtime -> Restart session, and re-run this cell. Supervisor "
        "Experiment 2 is not designed to run on CPU (9 epochs x 5 steps x 8 "
        "methods of CLIP-ViT training) and this notebook will not proceed "
        "without a GPU."
    )

print("CUDA version (torch build):", torch.version.cuda)
device_name = torch.cuda.get_device_name(0)
print("GPU name:", device_name)
total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU total memory: {total_mem_gb:.1f} GB")

try:
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,driver_version", "--format=csv"],
        check=False,
    )
except FileNotFoundError:
    print("(nvidia-smi not found on PATH -- torch's own CUDA report above is authoritative.)")


## 2. Mount Google Drive

Required for persistent output storage (Section 3) -- Colab will prompt for auth on first run.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 3. Persistent experiment root + resumable run ID

Defines the persistent Drive location for this experiment's logs/results/
cache/checkpoints, and a **stable run tag** (`EXP2_RUN_TAG`) that survives a
Colab disconnect: the tag is written once to
`EXPERIMENT_ROOT/logs/run_id.json` and reused on every subsequent run of this
notebook (against the same Drive folder) until you explicitly set
`FORCE_NEW_RUN_ID = True` to start a new logical attempt.

Reusing the same tag is what makes method-level resume work: it makes the
script's own `BASE_OUTPUT_DIR` (and therefore its `method_results/` folder)
land at the exact same path across reconnects, so already-completed methods
are detected and skipped instead of retrained -- see Section 9 and
`thesis_agent/reports/supervisor_exp2_colab_sync_audit.md`.

In [ ]:
import os
import json
import time

# Only this path is Colab/user-specific -- everything downstream is derived
# from it. Change it if you want a different Drive location for this
# experiment's outputs.
EXPERIMENT_ROOT = "/content/drive/MyDrive/vit-cifar100-project-runs/supervisor_exp2_rank32_re160"

SUBDIRS = ["logs", "results", "cache", "checkpoints"]
# NOTE on checkpoints/: the Exp2 script itself never writes model checkpoints
# to disk (every Trainer call site uses save_strategy="no" -- verified by
# reading the script, not assumed; best-epoch selection is done by reloading
# in-memory state, not from a saved checkpoint file). This directory is
# created for forward-compatibility / manual use only; it will stay empty
# during a normal Exp2 run.
for sub in SUBDIRS:
    os.makedirs(os.path.join(EXPERIMENT_ROOT, sub), exist_ok=True)

print("Persistent experiment root:", EXPERIMENT_ROOT)
for sub in SUBDIRS:
    print(" -", os.path.join(EXPERIMENT_ROOT, sub))

# Set to True to deliberately abandon any previous run_id.json and start a
# brand-new logical experiment attempt (a fresh BASE_OUTPUT_DIR, no methods
# treated as already-completed). Leave False to resume/continue the existing
# attempt, if any.
FORCE_NEW_RUN_ID = False

run_id_path = os.path.join(EXPERIMENT_ROOT, "logs", "run_id.json")

if os.path.exists(run_id_path) and not FORCE_NEW_RUN_ID:
    with open(run_id_path) as f:
        run_id_info = json.load(f)
    EXP2_RUN_TAG = run_id_info["run_tag"]
    print(f"\nReusing existing EXP2_RUN_TAG from a previous attempt: {EXP2_RUN_TAG}")
    print("(Any already-completed methods for this tag will be detected and skipped -- see Section 9.)")
else:
    EXP2_RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
    with open(run_id_path, "w") as f:
        json.dump({"run_tag": EXP2_RUN_TAG, "created_at": time.strftime("%Y-%m-%dT%H:%M:%S")}, f, indent=2)
    reason = "FORCE_NEW_RUN_ID=True" if FORCE_NEW_RUN_ID else "no previous run_id.json found"
    print(f"\nStarting a NEW logical experiment attempt ({reason}).")
    print(f"EXP2_RUN_TAG = {EXP2_RUN_TAG}")

os.environ["EXP2_RUN_TAG"] = EXP2_RUN_TAG
os.environ["EXP2_RESULTS_ROOT"] = os.path.join(EXPERIMENT_ROOT, "results")

print("\nos.environ['EXP2_RUN_TAG']     =", os.environ["EXP2_RUN_TAG"])
print("os.environ['EXP2_RESULTS_ROOT'] =", os.environ["EXP2_RESULTS_ROOT"])


## 4. Clone / update the GitHub repository

Clones into `/content/vit-cifar100-project` (ephemeral local disk -- code
only, not outputs). If already present, updates via `fetch` + `pull
--ff-only` (never force-overwrites local work; fails loudly instead of
silently diverging). Prints `git rev-parse HEAD` so the exact commit is
recorded in the Colab output, and **hard-fails if the working tree is
dirty** -- this notebook must never execute an uncommitted local version of
the script.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/Kasra-Shp/vit-cifar100-project.git"
REPO_DIR = "/content/vit-cifar100-project"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Cloning {REPO_URL} -> {REPO_DIR}")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"{REPO_DIR} already present -- updating (fetch + fast-forward only).")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
print("\nWorking directory:", os.getcwd())

commit_hash = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR).decode().strip()
print("Repository commit HEAD:", commit_hash)

status = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO_DIR).decode()
if status.strip():
    raise RuntimeError(
        "Working tree has uncommitted changes -- refusing to run an "
        "uncommitted version of Experiment 2. Commit/stash/discard these "
        "changes (in whatever remote/environment produced them) and re-run "
        "this cell, or push them to GitHub first if they are intentional:\n"
        + status
    )
print("Working tree is clean -- will execute exactly this committed code.")


## 5. Install dependencies

Inspects what Exp2 actually imports and installs only what Colab's base
image is missing. **Does not touch `torch`/`torchvision`** (Colab already
ships a CUDA build matched to its own driver -- reinstalling/upgrading them
is how Colab GPU environments usually get broken). No repo-level
`requirements.txt` exists to defer to (checked: none is present anywhere in
this repository), so versions are left unpinned and are instead verified by
successfully importing everything Exp2 needs, immediately below.

In [ ]:
import importlib.util
import subprocess
import sys

def is_installed(module_name):
    return importlib.util.find_spec(module_name) is not None

# import_name -> pip package name
CANDIDATE_PACKAGES = {
    "transformers": "transformers",
    "datasets": "datasets",
    "peft": "peft",
    # Optional: only used for smoother (PCHIP) live convergence-plot curves --
    # the script has a built-in fallback to plain polylines if absent (see
    # _HAVE_SCIPY in the script), so this is a nice-to-have, not required.
    "scipy": "scipy",
}

already_present = [name for name in ["numpy", "pandas", "matplotlib", "PIL", "torch", "torchvision"] if is_installed(name)]
print("Already present in this Colab image:", already_present)

to_install = [pip_name for mod_name, pip_name in CANDIDATE_PACKAGES.items() if not is_installed(mod_name)]
print("Packages to install:", to_install if to_install else "(none -- all already present)")

if to_install:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + to_install, check=True)
    print("Install complete.")


In [ ]:
import importlib

# Every top-level import Exp2's own script performs (read directly from its
# import cell) -- verifies the environment can actually run it, without
# running any of its training/eval code.
REQUIRED_MODULES = [
    "os", "gc", "json", "random", "math", "inspect", "datetime",
    "numpy", "pandas", "matplotlib.pyplot", "PIL",
    "torch", "torch.nn", "torch.nn.functional",
    "datasets", "torchvision",
    "transformers", "transformers.modeling_outputs",
    "peft",
]

failures = []
for mod in REQUIRED_MODULES:
    try:
        importlib.import_module(mod)
    except ImportError as e:
        failures.append((mod, str(e)))

if failures:
    raise ImportError(f"Missing/broken imports required by Exp2 after install: {failures}")
print("All imports required by Exp2 verified OK.")

try:
    import scipy.interpolate  # noqa: F401
    print("scipy available -- Exp2 will use smooth PCHIP interpolation for its live convergence plots.")
except ImportError:
    print("scipy NOT available -- Exp2 falls back to plain polylines for live convergence plots "
          "(cosmetic only: does not affect trained weights, evaluation, or any reported metric).")

import torch
import transformers
import peft
import datasets as hf_datasets

print()
print("torch:       ", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("peft:        ", peft.__version__)
print("datasets:    ", hf_datasets.__version__)


## 6. Dataset / model caching

CLIP-ViT-B/16 weights (~600MB) and CIFAR-100 (~170MB) are cached via the
standard Hugging Face `HF_HOME` environment variable -- **no change to the
Exp2 script itself** is needed for this (it already relies on the HF
libraries' own default cache resolution, which respects these variables
automatically).

**Default here: ephemeral `/content` local disk** -- fast (avoids Drive's
FUSE-mount I/O overhead for the many small files HF caching touches), but
lost on VM recycle/disconnect. Re-downloading after a fresh VM costs a few
minutes, **once per VM, not per method** -- unrelated to and unaffected by
the method-level resume mechanism in Section 9 (resume is about not
*retraining* a completed method, not about not *redownloading* the model).

If you would rather trade I/O speed for cross-session cache persistence,
uncomment the `EXPERIMENT_ROOT`-based line below instead.

In [ ]:
import os

# EPHEMERAL (default): fast local disk, lost on VM recycle.
CACHE_ROOT = "/content/hf_cache"

# PERSISTENT (alternative): uncomment to cache on Drive instead -- slower
# (FUSE-mounted I/O) but survives a full VM loss, so a fresh VM does not need
# to re-download CLIP/CIFAR-100 at all.
# CACHE_ROOT = os.path.join(EXPERIMENT_ROOT, "cache")

os.makedirs(CACHE_ROOT, exist_ok=True)
os.environ["HF_HOME"] = CACHE_ROOT
os.environ["HF_DATASETS_CACHE"] = os.path.join(CACHE_ROOT, "datasets")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(CACHE_ROOT, "transformers")

print("HF_HOME:            ", os.environ["HF_HOME"], "(ephemeral)" if CACHE_ROOT == "/content/hf_cache" else "(persistent, on Drive)")
print("HF_DATASETS_CACHE:  ", os.environ["HF_DATASETS_CACHE"])
print("TRANSFORMERS_CACHE: ", os.environ["TRANSFORMERS_CACHE"])


## 7. Reproducibility record

Writes a small JSON manifest to persistent Drive storage recording everything
the *Colab session* knows that the script itself does not (git commit, GPU,
library versions, wall-clock start time, output directory). The script's own
`configs/run_config.json` (written inside `EXP2_RESULTS_ROOT/.../configs/`
when it runs) separately records every scientific constant (seed, rank,
schedule, KD/FactorOrth params, etc.) read live from the executing process --
this manifest complements, not duplicates, that file.

In [ ]:
import json
import time
import torch

reproducibility_manifest = {
    "experiment": "supervisor_exp2_cifar100_5x20_rank32_vs_rankext160",
    "script_path": "experiments_prepared/supervisor_exp2_cifar100_5x20_rank32_vs_rankext160.py",
    "git_commit": commit_hash,
    "exp2_run_tag": EXP2_RUN_TAG,
    "exp2_results_root": os.environ["EXP2_RESULTS_ROOT"],
    "seed": 42,
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_total_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 1),
    "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda,
    "python_version": sys.version,
    "start_time": time.strftime("%Y-%m-%dT%H:%M:%S"),
}

manifest_path = os.path.join(EXPERIMENT_ROOT, "logs", f"colab_session_manifest_{EXP2_RUN_TAG}.json")
with open(manifest_path, "w") as f:
    json.dump(reproducibility_manifest, f, indent=2)

print(json.dumps(reproducibility_manifest, indent=2))
print("\nWritten to:", manifest_path)


## 8. GPU memory risk (static assessment -- no scientific settings reduced)

Exp2 trains CLIP-ViT-B/16 (~86M backbone params) with LoRA adapters on
`q_proj`/`v_proj` only, up to RankExt's final cumulative rank 160
(SimpleAvg stays at rank 32). LoRA parameter counts at these ranks are
tiny relative to the frozen backbone (a rank-160, 2-module, 12-layer LoRA
adapter is on the order of a few million parameters, vs. ~86M frozen
backbone params) -- rank width is not expected to be the memory bottleneck.
Batch size is 16 (`BATCH_LORA`, unchanged), and the script already trains
under `fp16=True` whenever CUDA is available (`USE_FP16 = torch.cuda.is_available()`,
wired into every `TrainingArguments` call -- **this notebook does not add or
change that**, it was already present in the audited Exp2 script). Every
method also explicitly frees GPU memory before the next one starts
(`del <model>; gc.collect(); torch.cuda.empty_cache()`).

**Assessment: LOW risk** on any Colab GPU tier (T4/16GB and above) given the
small backbone, existing fp16 training, small per-step batch, and explicit
inter-method cleanup. No rank, batch size, model, precision, or method count
was reduced to reach this conclusion -- if a real OOM is observed in
practice, that is a finding to report back, not something to silently paper
over by shrinking the experiment.

## 9. Run exact Supervisor Experiment 2

**This is the only cell that starts real training.** It runs the committed
script exactly as-is (`python -u experiments_prepared/supervisor_exp2_cifar100_5x20_rank32_vs_rankext160.py`)
from the repository root, with no notebook-local method/hyperparameter
overrides. `stdout`/`stderr` are captured with `tee` into a log file under
persistent Drive storage.

**Resume behavior:** `EXP2_RUN_TAG`/`EXP2_RESULTS_ROOT` (Section 3) make the
script's own `BASE_OUTPUT_DIR` -- and therefore its
`method_results/<method>.json` atomic per-method result files -- land at the
same Drive path across reconnects. If this cell is re-run after a Colab
disconnect (same notebook, same `EXPERIMENT_ROOT`, `FORCE_NEW_RUN_ID` still
`False`), any of the 8 methods that already finished will be detected
(config-fingerprint-checked) and **skipped instead of retrained**; only the
method that was interrupted (or never started) actually trains. This is
implemented inside the script itself (method-level only -- no batch/epoch
resume), so it is identical infrastructure whether this script is launched
from Colab or from the cluster.

In [ ]:
import os
import time

SCRIPT_PATH = "experiments_prepared/supervisor_exp2_cifar100_5x20_rank32_vs_rankext160.py"
assert os.path.isfile(SCRIPT_PATH), f"{SCRIPT_PATH} not found -- run this from the repo root ({REPO_DIR})"

# Matches the sbatch script's own "unset REPLICATION_SEED || true" convention:
# make sure nothing in the Colab environment can override the script's own
# canonical seed default (42) via this env var.
os.environ.pop("REPLICATION_SEED", None)

log_path = os.path.join(EXPERIMENT_ROOT, "logs", f"exp2_colab_{EXP2_RUN_TAG}.log")

print("=" * 80)
print("LAUNCHING SUPERVISOR EXPERIMENT 2")
print("=" * 80)
print("Repository commit: ", commit_hash)
print("Seed:              42 (REPLICATION_SEED unset -> script's own canonical default)")
print("GPU:               ", torch.cuda.get_device_name(0))
print("PyTorch / CUDA:    ", torch.__version__, "/", torch.version.cuda)
print("EXP2_RUN_TAG:      ", EXP2_RUN_TAG)
print("EXP2_RESULTS_ROOT: ", os.environ["EXP2_RESULTS_ROOT"])
print("Log file:          ", log_path)
print("Start time:        ", time.strftime("%Y-%m-%dT%H:%M:%S"))
print("=" * 80)

exit_code = os.system(f'python -u "{SCRIPT_PATH}" 2>&1 | tee "{log_path}"')
print("\nExit code:", exit_code)


## 10. If Colab disconnects

1. Reconnect the runtime (Runtime -> Reconnect, or open the notebook again).
2. Re-run every cell from the top, in order, **with `FORCE_NEW_RUN_ID` still
   `False`** in Section 3. Sections 1-8 are cheap/idempotent (GPU check,
   Drive mount, repo update, dependency install/verify, cache/manifest
   setup); they do not repeat any training.
3. Section 3 will print `Reusing existing EXP2_RUN_TAG from a previous
   attempt: <tag>` -- confirming resume is active for this attempt.
4. Re-run Section 9. The script will print, for each already-completed
   method, `[resume] SKIPPED training for <method> -- reloaded a previously
   completed, config-matched result from .../method_results/<method>.json`,
   then continue training only the remaining method(s).

If you deliberately want to discard a previous attempt and start over from
method 1, set `FORCE_NEW_RUN_ID = True` in Section 3 once, then set it back
to `False` for subsequent reconnects within that new attempt.